**Dataset Preparation**

Loads the partitioned CSVs (`4-shot.csv`, `8-shot.csv`, `eval.csv`) from `/partitioned_data` and their corresponding renders from `/renders`, and saves each split as a HuggingFace `Dataset` to `OUT_DIR`. Missing renders are audited and reported before loading; any affected rows are dropped cleanly rather than propagated downstream.

In [ ]:
import os
import pandas as pd
from pathlib import Path
from datasets import Dataset, Features, Value, Image as HFImage
from PIL import Image as PILImage

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
IN_DIR  = Path("/scratch/gpfs/FELLBAUM/af3158/cos351/mcd_dataset")
OUT_DIR = Path("/scratch/gpfs/FELLBAUM/af3158/cos351/open_weight_n_shot/hf_dataset")

In [ ]:
RENDERS_DIR      = IN_DIR  / "renders"
PARTITIONED_DIR  = IN_DIR  / "partitioned_data"

SPLITS = {
    "4shot": PARTITIONED_DIR / "4-shot.csv",
    "8shot": PARTITIONED_DIR / "8-shot.csv",
    "eval":  PARTITIONED_DIR / "eval.csv",
}

# ── Features schema ──────────────────────────────────────────────────────────
features = Features({
    "image":        HFImage(),
    "file_name":    Value("string"),
    "label":        Value("string"),
    "description":  Value("string"),
    "key_comments": Value("string"),
    "source_url":   Value("string"),
    "drive_link":   Value("string"),
})

# ── Build & save each split ───────────────────────────────────────────────────
missing_log = {}

for split_name, csv_path in SPLITS.items():
    print(f"\n── Processing {split_name} ({csv_path.name}) ──")
    df = pd.read_csv(csv_path)

    # Audit for missing renders before loading
    missing = [fn for fn in df["file_name"] if not (RENDERS_DIR / fn).exists()]
    if missing:
        missing_log[split_name] = missing
        print(f"  ⚠  {len(missing)} missing render(s): {missing[:5]}{'...' if len(missing) > 5 else ''}")

    # Drop missing rows rather than crashing
    df = df[df["file_name"].apply(lambda fn: (RENDERS_DIR / fn).exists())].reset_index(drop=True)
    print(f"  ✓  {len(df)} valid rows")

    # Load images
    images = []
    for fn in df["file_name"]:
        img_path = RENDERS_DIR / fn
        try:
            img = PILImage.open(img_path).convert("RGB")
            images.append(img)
        except Exception as e:
            print(f"  ✗  Failed to open {fn}: {e}")
            images.append(None)

    data_dict = {
        "image":        images,
        "file_name":    df["file_name"].tolist(),
        "label":        df["label"].tolist(),
        "description":  df["description"].fillna("").tolist(),
        "key_comments": df["key_comments"].fillna("").tolist(),
        "source_url":   df["source_url"].fillna("").tolist(),
        "drive_link":   df["drive_link"].fillna("").tolist(),
    }

    dataset = Dataset.from_dict(data_dict, features=features)

    out_path = OUT_DIR / split_name
    out_path.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(out_path))
    print(f"  ✓  Saved → {out_path}  ({len(dataset)} examples)")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n══ Done ══")
for split_name, csv_path in SPLITS.items():
    out_path = OUT_DIR / split_name
    print(f"  {split_name:6s}  →  {out_path}")

if missing_log:
    print("\n⚠  Missing renders summary:")
    for split, files in missing_log.items():
        print(f"  {split}: {files}")
else:
    print("\n✓  No missing renders detected")